In [ ]:
import itertools
import concurrent.futures
import constants as Cs
import os
import json
import datetime
import pickle
import optuna
from constants import EXAMPLE_MAPPING
SEEDS = list(range(101, 115))
TEST_EVAL_EPS = 5
# lambda fit archving [FrozenTrial(number=80, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 4, 53, 822616), datetime_complete=datetime.datetime(2026, 5, 22, 5, 36, 8, 359073), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=81, value=None), FrozenTrial(number=86, state=<TrialState.COMPLETE: 1>, values=[118.11550385203829], datetime_start=datetime.datetime(2026, 5, 22, 5, 36, 31, 908497), datetime_complete=datetime.datetime(2026, 5, 22, 6, 11, 5, 128793), params={'crossmethod': 'uniform', 'lambda': 70, 'mu': 70, 'mutation_rate': 0.01, 'cross_rate': 0.9000000000000001, 'sigma': 0.5, 'archiving_period': 4, 'archive_batch': 5, 'cross': 0.6}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.1, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5), 'archiving_period': IntDistribution(high=5, log=False, low=2, step=1), 'archive_batch': IntDistribution(high=5, log=False, low=1, step=1), 'cross': FloatDistribution(high=0.9, log=False, low=0.1, step=0.1)}, trial_id=87, value=None)]
# lambda - this one is bad actually novelty [FrozenTrial(number=4, state=<TrialState.COMPLETE: 1>, values=[-9.291787437438707], datetime_start=datetime.datetime(2026, 5, 19, 22, 25, 13, 886515), datetime_complete=datetime.datetime(2026, 5, 19, 23, 0, 16, 677521), params={'crossmethod': 'mean', 'lambda': 60, 'mu': 60, 'mutation_rate': 0.18, 'cross_rate': 1.0, 'sigma': 2.5}, user_attrs={}, system_attrs={}, intermediate_values={}, distributions={'crossmethod': CategoricalDistribution(choices=('uniform', 'mean')), 'lambda': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mu': CategoricalDistribution(choices=(40, 50, 60, 70)), 'mutation_rate': FloatDistribution(high=0.5, log=False, low=0.0, step=0.01), 'cross_rate': FloatDistribution(high=1.0, log=False, low=0.3, step=0.1), 'sigma': FloatDistribution(high=3.0, log=False, low=0.5, step=0.5)}, trial_id=208, value=None)]

GENS = {
    "cartpole": [25],
    "lunarlander":[50]
}
DECAYS = {
    "lunarlander":{
        "diff":[0.2, 0.5, 1.0, 1.5],
        "lambda":[0.5, 1.0, 2.0, 3.0]}
}

def rename(dict):
    return {EXAMPLE_MAPPING.get(k, k): v for k, v in dict.items()}

def task_job(en, alg, container, args, s, out_path):
    env = Cs.ENIVROMENTS[en]()
    df, pop = Cs.ALG_MAPPING[alg].argumented_function(env=en, container=container, seed=s, out_path=out_path, **args)
    print("Testing " + str(s))
    fitnesses = [env.evalutation_b(p, 42, TEST_EVAL_EPS) for p in pop]
    print("Finished seed %d of algorithm %s" % (s, alg))
    return fitnesses, pop


def evaluation_of_setup(en, alg, container, out_path, **kwargs):
    #we do not deviate the sigma
    # first we evaluate the current setup

    stat_futures = {}
    gens_future={}
    dec_future = {}
    args = rename(kwargs)
    dirpath = os.path.join(os.path.realpath(out_path), container,alg)

    with concurrent.futures.ProcessPoolExecutor(max_workers=5) as executor:
        print("Launching " + alg + "on Enviroment " + str(en))
        for dec, ng, s in itertools.product(DECAYS[en][alg], GENS[en], SEEDS):
            ags = args.copy()
            ags["ng"] = ng
            ags["decay"] = dec
            future = executor.submit(task_job, container=container, alg=alg, en=en, args=args, s=s, out_path=dirpath + "/stat")
            stat_futures[future] = s
            dec_future[future] = dec
            gens_future[future] = ng


    stats = {}
    pops = {}
    for future in concurrent.futures.as_completed(stat_futures):
        s = stat_futures[future]
        ng = gens_future[future]
        dec = dec_future[future]

        result = future.result()
        if "fitness" not in stats:
            stats["fitness"] = {}
        if dec not in stats["fitness"]:
            stats["fitness"][dec] = {}
        if ng not in stats["fitness"][dec]:  
            stats["fitness"][dec][ng] = {}

        stats["fitness"][dec][ng][s] = result[0]
        pops[s]= result[1] 
    ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    filename = f"{ts}_{container}_"+ "|".join(f"{k}: {v}" for k, v in args.items())
    stats["environment"] = en
    stats["algorithm"] = alg
    stats["container"] = container
    stats["arguments"] = args
    json_path = os.path.join(dirpath, filename + ".json")
    #pickle_path = os.path.join(dirpath, filename + ".plk")

    with open(json_path, "w") as json_file:
        json.dump(stats,json_file)
        print("Finished "+ filename)
    #with open(pickle_path, "wb") as f:
    #    pickle.dump(pops, f)
    return pops, stats



In [ ]:
import numpy as np

def select_minimal_exaples(examples):
    pop = np.inf
    selected_trials = []
    for t in examples:
        k = t["lambda"]+t["mu"]
        if pop == k:
            selected_trials.append(t)
        elif pop > k:
            pop = k
            selected_trials = [t]

    return selected_trials


def make_decay_examination(en, alg, container):
    with open("relevant_studies.json", "r") as f:
        relevant_study_names = json.load(f)
    storage = f"sqlite:///Data/optuna/{en}/{container}/{alg}.db"
    study_name = relevant_study_names[container][alg][en]

    new_study = optuna.load_study(study_name=study_name,storage=storage)
    most_promising = select_minimal_exaples([t.params for t in new_study.best_trials])
    pops, stats = evaluation_of_setup(
        en=en, 
        alg=alg, 
        container=container,
        out_path=f"./Data/decay_search/{en}",
        **most_promising[0]
    )
    return pops, stats



Launching lambdaon Enviroment lunarlander
   	      	                                fitness                                	                                novelty                                 
   	      	-----------------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg     	gen	max     	min     	nevals	std    	avg     	gen	max     	min        	nevals	std     
0  	70    	-383.202	0  	-125.803	-616.413	70    	144.004	0.401909	0  	0.800295	0.000330882	70    	0.223086
   	      	                                fitness                                	                                novelty                                 
   	      	-----------------------------------------------------------------------	------------------------------------------------------------------------
gen	nevals	avg     	gen	max     	min     	nevals	std    	avg     	gen	max     	min      	nevals	std     
0  	70    	-391

In [38]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
def load_direct_data(en, cont, alg, name):
    path = f"./Data/decay_search/{en}/{cont}/{alg}/{name}"
    json_files = list(Path(path).glob("*.json"))
    print(f"Found {len(json_files)} files!")
    with open(json_files[0], "r", encoding="utf-8") as f:
        data = json.load(f)
    return data

def extract_fitness_values(file_content):
    df  = pd.json_normalize(file_content, sep="/")
    df = df.melt(var_name="path", value_name="population")
    df = df[df["path"].str.contains("fitness/")]
    df[["type", "decay","ng", "seed"]] = (df["path"].str.split("/", expand=True))
    df = df.drop(columns=["path","type"])
    df["ng"] = df["ng"].astype(int)
    df["max_fit"] = df["population"].apply(lambda x: np.max(list(map(lambda y: y[0], x))))
    df["avg_fit"] = df["population"].apply(lambda x: np.mean(list(map(lambda y: y[0], x))))
    df["median_fit"] = df["population"].apply(lambda x: np.median(list(map(lambda y: y[0], x))))
    df["min_fit"] = df["population"].apply(lambda x: np.min(list(map(lambda y: y[0], x))))
    df["std_fit"] = df["population"].apply(lambda x: np.std(list(map(lambda y: y[0], x))))
    return df
def save_final_hyperparameters(en, cont, alg, hyper_paramters):
    
    with open("relevant_finals.json", "r+", encoding="utf-8") as f:
        try:
            data = json.load(f)
        except json.JSONDecodeError:
            data = {}

        f.seek(0)       
        if cont not in data:
            data[cont] = {}
        if en not in data[cont]:
            data[cont][en] = {}
        data[cont][en][alg] = hyper_paramters
        json.dump(data, f)
        f.truncate()       


In [39]:
en = "lunarlander"
cont = "add_novelty"
alg = "lambda"
file_content = load_direct_data(en, cont, alg, "")
df = extract_fitness_values(file_content)

df

Found 1 files!


,population,decay,ng,seed,max_fit,avg_fit,median_fit,min_fit,std_fit
3,"[[[-107.35161482644148], [0.40035834312438967,...",1.0,50,103,-107.351615,-107.351615,-107.351615,-107.351615,2.842171e-14
4,"[[[-79.25048589069709], [0.018930283188819886,...",1.0,50,108,-79.250486,-79.250486,-79.250486,-79.250486,2.842171e-14
5,"[[[-95.64327179150544], [0.1386697769165039, -...",1.0,50,102,-95.643272,-95.643272,-95.643272,-95.643272,2.842171e-14
6,"[[[-328.4736325811808], [-0.24376974105834961,...",1.0,50,104,-328.473633,-328.473633,-328.473633,-328.473633,1.136868e-13
7,"[[[-171.51388775824657], [-0.10178447961807251...",1.0,50,107,-171.513888,-171.513888,-171.513888,-171.513888,0.000000e+00
8,"[[[-191.64825796563696], [0.3218778133392334, ...",1.0,50,112,-121.630796,-330.185053,-352.991336,-352.991336,5.208695e+01
9,"[[[-301.45674998667675], [0.08144283294677734,...",1.0,50,109,-301.456750,-301.456750,-301.456750,-301.456750,1.705303e-13
10,"[[[-128.25811319259498], [0.2876248836517334, ...",1.0,50,111,-128.258113,-128.258113,-128.258113,-128.258113,5.684342e-14
11,"[[[-314.89217576047486], [-0.38494441509246824...",1.0,50,101,-314.892176,-314.892176,-314.892176,-314.892176,1.705303e-13
12,"[[[77.39966133256887], [0.09966484308242798, -...",1.0,50,110,77.399661,77.399661,77.399661,77.399661,0.000000e+00


In [49]:
import seaborn as sns
df = df.sort_values("decay")
ploted_df = df[["decay", "seed", "max_fit", "median_fit", "avg_fit"]]
ploted_df = ploted_df.groupby("decay").max()
ploted_df
# if "seed" in ploted_df.columns:
#     ploted_df = ploted_df.drop(columns="seed")
# ploted_df = ploted_df.melt(
#         id_vars="decay", 
#         var_name="type", 
#         value_name="value"
#     )
# sns.lineplot(data=ploted_df, x="decay", y="value", hue="type")

,seed,max_fit,median_fit,avg_fit
decay,,,,
0.5,114,77.399661,77.399661,77.399661
1.0,114,77.399661,77.399661,77.399661
2.0,114,77.399661,77.399661,77.399661
3.0,114,77.399661,77.399661,77.399661
